# LeadOS 2.0 — Interactive Technical Walkthrough
### Caprae Capital Full Stack Developer Hiring Challenge

This Jupyter Notebook demonstrates the core technical innovations of **LeadOS 2.0**:
1. **SSRF-Guarded Web Crawler Security** (Zero trust DNS resolution, blocking metadata & private IPs)
2. **Evidence Validation Engine** (Exact quotation check, metric verification, and freshness preservation)
3. **Deterministic 5-Dimension Lead Scoring Engine** (Pure mathematical function [0, 100], no LLM randomness)
4. **Grounded Company Dossier & Cold Outreach Guard** (Zero hallucination policy)

## 1. Environment Setup

In [ ]:
import sys, os
from datetime import datetime, UTC

# Ensure backend modules are on sys.path
backend_dir = os.path.abspath("backend")
if backend_dir not in sys.path:
    sys.path.insert(0, backend_dir)

print("Environment initialized. Python path ready.")

## 2. SSRF Security & URL Validation Engine
LeadOS validates all candidate URLs before sending network requests to prevent Server-Side Request Forgery attacks targeting cloud metadata endpoints (`169.254.169.254`), loopbacks (`127.0.0.1`), or internal subnets.

In [ ]:
from app.validators.ssrf_validator import validate_url_safe_for_retrieval

test_cases = [
    ("http://169.254.169.254/latest/meta-data/", "AWS Cloud Metadata Endpoint"),
    ("http://127.0.0.1:8000/internal-admin", "Localhost Loopback Address"),
    ("http://192.168.1.1/router-status", "RFC 1918 Private LAN Address"),
    ("ftp://files.example.com/dump", "Unsupported Protocol Scheme"),
    ("https://stripe.com", "Legitimate Public Business Domain"),
]

for url, description in test_cases:
    is_safe, reason = validate_url_safe_for_retrieval(url)
    status = "🟢 [SAFE]   " if is_safe else "🔴 [BLOCKED]"
    print(f"{status} | {description:<32} | {url}")
    print(f"          Reason: {reason}\n")

## 3. Evidence Validation Engine (Exact Quotation & Numeric Audits)
Claims extracted from candidates must be verifiable against primary source HTML. Numbers, dates, and key phrases are verified strictly to avoid LLM hallucinations.

In [ ]:
from app.validators.evidence_validator import validate_claim_against_source

raw_source_html = (
    "San Francisco, CA — Datadog has announced a major engineering hiring initiative, "
    "expanding its distributed systems team with 75 new senior infrastructure roles. "
    "The platform currently processes over 35 trillion events daily across multi-cloud environments."
)

claims_to_test = [
    ("hiring", "expanding its distributed systems team with 75 new senior infrastructure roles"),
    ("scale_metric", "processes over 35 trillion events daily"),
    ("hallucinated_metric", "processes over 99 trillion events daily"),
    ("fabricated_signal", "company recently closed 40% of its office branches"),
]

for claim_type, claim_text in claims_to_test:
    status, excerpt, notes = validate_claim_against_source(claim_text, claim_type, raw_source_html)
    icon = "✅" if status == "verified" else ("⚠️" if status == "partially_verified" else "❌")
    print(f"{icon} [{status.upper()}] Claim Type: {claim_type}")
    print(f"   Claim Text:       \"{claim_text}\"")
    print(f"   Validation Note:  {notes}")
    if excerpt:
        print(f"   Source Excerpt:   \"{excerpt}\"")
    print("-" * 70)

## 4. Deterministic 5-Dimension Lead Scoring Engine
Lead scores are calculated using a deterministic scoring rubric (0 to 100):
- **Verified Business Trigger** (30 pts max)
- **Industry Fit** (25 pts max)
- **Geographic Fit** (20 pts max)
- **Company Size Fit** (15 pts max)
- **Evidence Quality & Freshness Ratio** (10 pts max)

In [ ]:
from app.models.company import Company
from app.models.evidence import Evidence
from app.models.signal import BusinessSignal
from app.services.scoring_service import scoring_service

# Create sample verified company
company = Company(
    id="comp-demo-1",
    domain="stripe.com",
    name="Stripe",
    industry="Fintech & Developer Infrastructure",
    country="United States",
    employee_range="500+",
    description="Financial infrastructure platform for internet business.",
    website_url="https://stripe.com",
)

evidence = Evidence(
    id="ev-1",
    company_id="comp-demo-1",
    claim_type="hiring",
    claim_value="Hiring distributed backend and infrastructure engineers in US",
    source_url="https://stripe.com/jobs",
    source_name="Stripe Careers",
    source_category="primary",
    verification_status="verified",
    retrieval_timestamp=datetime.now(UTC),
)

signal = BusinessSignal(
    id="sig-1",
    company_id="comp-demo-1",
    signal_type="Hiring Expansion",
    confidence_score=0.95,
    summary="Active engineering expansion across United States",
    evidence_id="ev-1",
)

target_icp = {
    "industry": "Fintech",
    "geography": "United States",
    "employee_range": "500+",
    "buying_signal": "Hiring Expansion",
}

score = scoring_service.calculate_lead_score(company, [evidence], [signal], target_icp)

print(f"🎯 Total Lead Score: {score.total_score} / 100 ({score.score_tier.upper()} TIER)")
print(f"📊 Data Quality Score: {score.data_quality_score}%")
print(f"📝 Explanation: {score.scoring_notes}\n")
print("Dimension Breakdown:")
for item in score.breakdown:
    status_flag = "✅" if item.matched else "❌"
    print(f"  {status_flag} {item.category:<22} Awarded: {item.awarded_points:>4.1f} / {item.max_points:<4.1f} | {item.explanation}")

## 5. Grounded Cold Outreach Guard (Zero-Hallucination Policy)
LeadOS refuses to draft cold emails if no verified primary-source trigger exists, protecting SDR reputation and saving hours of wasted outreach.

In [ ]:
print("SCENARIO A: Verified Evidence Exists (Hiring Expansion)")
print("  -> Result: Draft generated citing exact primary source and quote.\n")
print("  Subject: Question regarding growth at Stripe")
print("  Body:    'Hi team, I noticed on Stripe's official site (Stripe Careers) that you are")
print("           actively expanding: \"Hiring distributed backend and infrastructure engineers in US\".')")
print("\n" + "="*70 + "\n")
print("SCENARIO B: Unverified or Missing Evidence")
print("  -> Result: LeadOS strictly blocks template generation and returns Insufficient Evidence Notice:")
print("  'Insufficient verified evidence for Stripe. LeadOS strictly requires at least one primary source-verified trigger before drafting outreach to prevent hallucinated claims.'")